In [3]:

import requests, urllib.parse, json, re, unicodedata, time
from threading import Lock, Thread
from queue import Queue

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HDR = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# Poprawna polska transliteracja
PL_MAP = str.maketrans("ąćęłńóśźżĄĆĘŁŃÓŚŹŻ", "acelnoszzACELNOSZZ")

def slugify_pl(text):
    text = text.translate(PL_MAP)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    text = re.sub(r"[^\w\s-]", "", text.lower())
    text = re.sub(r"[\s_-]+", "-", text).strip("-")
    return text[:96]

# Test
tests = [("Gładź gipsowa Knauf G-K Finish 25 kg", "gladz-gipsowa-knauf-g-k-finish-25-kg"),
         ("Zaprawa klejąca Atlas Hoter U 25 kg", "zaprawa-klejaca-atlas-hoter-u-25-kg"),
         ("Farba biała do ścian", "farba-biala-do-scian")]
print("Test slugify_pl:")
for name, expected in tests:
    got = slugify_pl(name)
    ok = "✅" if got == expected else "❌"
    print(f"  {ok} '{name[:40]}' → {got}")


Test slugify_pl:
  ✅ 'Gładź gipsowa Knauf G-K Finish 25 kg' → gladz-gipsowa-knauf-g-k-finish-25-kg
  ✅ 'Zaprawa klejąca Atlas Hoter U 25 kg' → zaprawa-klejaca-atlas-hoter-u-25-kg
  ✅ 'Farba biała do ścian' → farba-biala-do-scian


In [7]:

import requests, urllib.parse, json, re, unicodedata, time
from threading import Lock, Thread
from queue import Queue

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
HDR_JSON = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
HDR_GET  = {"Authorization": f"Bearer {TOKEN}"}

PL_MAP = str.maketrans("ąćęłńóśźżĄĆĘŁŃÓŚŹŻ", "acelnoszzACELNOSZZ")

def slugify_pl(text):
    text = text.translate(PL_MAP)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    text = re.sub(r"[^\w\s-]", "", text.lower())
    text = re.sub(r"[\s_-]+", "-", text).strip("-")
    return text[:96]

def sanity_query(q):
    r = requests.get(
        f"https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production?query={urllib.parse.quote(q)}",
        headers=HDR_GET, timeout=30)
    return r.json().get("result", [])

def sanity_mutate(muts):
    r = requests.post(
        "https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production",
        json={"mutations": muts}, headers=HDR_JSON, timeout=60)
    return r.json()

# Pobierz wszystkie produkty (id + nazwa + obecny slug)
print("Pobieram produkty z Sanity...")
all_prods = []
offset = 0
while True:
    batch = sanity_query(f'*[_type=="product"][{offset}..{offset+999}]{{_id,name,"slug":slug.current}}')
    all_prods.extend(batch)
    print(f"  Pobrano {len(all_prods)}...", flush=True)
    if len(batch) < 1000:
        break
    offset += 1000

print(f"Łącznie: {len(all_prods)} produktów")

# Wygeneruj nowe slugi
needs_fix = []
for p in all_prods:
    name = (p.get("name") or "").strip()
    if not name:
        continue
    new_slug = slugify_pl(name)
    old_slug = p.get("slug", "")
    if new_slug != old_slug:
        needs_fix.append({"_id": p["_id"], "slug": new_slug})

print(f"Produktów wymagających nowego sluga: {len(needs_fix)}")
if needs_fix:
    print(f"Przykład: '{all_prods[0].get('name','')}' → '{slugify_pl(all_prods[0].get('name',''))}'")


Pobieram produkty z Sanity...


  Pobrano 1000...


  Pobrano 2000...


  Pobrano 3000...


  Pobrano 4000...


...

  Pobrano 13000...


  Pobrano 14000...


  Pobrano 15000...


  Pobrano 15836...


Łącznie: 15836 produktów
Produktów wymagających nowego sluga: 6189
Przykład: 'Grunt Kwarcowy Ceresit Ct 16 Pod Tynki 10 L' → 'grunt-kwarcowy-ceresit-ct-16-pod-tynki-10-l'


In [11]:

fixed = 0
BATCH = 200
t0 = time.time()
for i in range(0, len(needs_fix), BATCH):
    batch = needs_fix[i:i+BATCH]
    muts = [{"patch": {"id": p["_id"], "set": {"slug": {"_type": "slug", "current": p["slug"]}}}} for p in batch]
    sanity_mutate(muts)
    fixed += len(batch)
    if fixed % 1000 == 0:
        print(f"  {fixed}/{len(needs_fix)} naprawionych...", flush=True)

elapsed = int(time.time()-t0)
print(f"\n✅ Naprawiono {fixed} slugów w {elapsed}s")

# Weryfikacja
sample = sanity_query('*[_type=="product" && sku in ["P-0009351","P-0026479","P-0253747"]]{"slug":slug.current,name}')
print("\nWeryfikacja:")
for p in sample:
    print(f"  {p['name'][:50]:50s} → {p['slug']}")


  1000/6189 naprawionych...


  2000/6189 naprawionych...


  3000/6189 naprawionych...


  4000/6189 naprawionych...


  5000/6189 naprawionych...


  6000/6189 naprawionych...



✅ Naprawiono 6189 slugów w 60s

Weryfikacja:
  Zaprawa klejąca Atlas Hoter U do styropianu i XPS  → zaprawa-klejaca-atlas-hoter-u-do-styropianu-i-xps-oraz-siatki-bialy-25-kg
  Gładź gipsowa Knauf G-K Finish 25 kg               → gladz-gipsowa-knauf-g-k-finish-25-kg
  Zaprawa uszczelniająca Atlas WODER SX do izolacji  → zaprawa-uszczelniajaca-atlas-woder-sx-do-izolacji-fundamentow-25-kg
